In [ ]:
# ============================================================================
# CELL 1: CONFIGURATION & IMPORTS
# ============================================================================

import re, json, time
import os
import pickle
from pathlib import Path
from typing import Dict, Any, List, Optional, Tuple
from datetime import datetime

import pandas as pd
import tiktoken
import openai
from openai import AsyncOpenAI

# =========================
# Path Configuration
# =========================

TRANSCRIPT_ROOT = Path("Transcripts_Whisper_order by patient")
PATIENT_PROFILE_CSV = Path("patient_profiles copy.csv")
FORM_ROOT = Path("data/Visiting Form/Visiting Form - Order by Patient")
EMBEDDINGS_CACHE_FILE = Path("cache/original_rag_embeddings_cache_whisper.pkl")  # Persistent cache

OUT_DIR = Path("original_rag_baseline_output_whisper")
OUT_DIR.mkdir(parents=True, exist_ok=True)
EMBEDDINGS_CACHE_FILE.parent.mkdir(parents=True, exist_ok=True)

# =========================
# API Configuration
# =========================

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]

# =========================
# Model Configuration
# =========================

MODEL_SUMMARY = "gpt-4o"           
MODEL_EMBED = "text-embedding-3-small"
TEMP_SUMMARY = 0.3                  

# =========================
# RAG Configuration
# =========================

TOP_K_RETRIEVAL = 3  # Number of previous visits to retrieve
SIMILARITY_THRESHOLD = 0.0  # No threshold - retrieve from all patients

# =========================
# Official Form Template
# =========================

OFFICIAL_FORM_TEMPLATE = """
1.【利用者の状態（病状・トピックス）】
このセクションは以下の内容で構成されています。

- トピックス: 
  今回の診察で最も重要な臨床観察について記述してください。
判断基準：
  1. 安全リスク: 転倒、出血、強い痛み、発熱（38℃以上）、意識変容、呼吸困難
  2. 症状の変化: 新たな症状、既存症状の悪化・改善、処置の変更
  3. 日常生活への影響: ADL低下、活動制限、睡眠障害
  4. 変化なし: 上記が該当しない場合のみ「特に重大な問題なし」
  
  【記載方法】
  - 具体的に記載
  - 時系列を明記: 「いつ」「何が」「どう変化したか」を含める
  - 臨床的意義: 日常生活や今後のケアへの影響を示唆

- フォームから抽出した各カテゴリーを【..】内にリストアップしてください。各カテゴリごとに、観察した事実を**臨床的文脈**と共に説明します。
  【記載ルール】
  1. 単なる事実の羅列ではなく、臨床的に意味のある文章で記載
  2. 複数の観察を関連付けて記載（例: 睡眠と疼痛の関係）
  3. 時系列や因果関係があれば明記
  4. 患者の主観的訴えと客観的観察を区別

- <処置内容・訓練内容>: 会話に基づき、実施した処置や訓練の内容を記載。
- <備考>: 特別な注意事項や重要な補足情報を記載。
- <外部連絡・伝達事項>: 外部機関（医師、ケアマネージャー、家族など）への連絡事項を記載。
- <内部連絡・伝達事項>: 事業所内のスタッフ間での申し送り事項を記載。

2.【実施した看護・リハビリテーションの内容】
※会話から推測できる実施内容を記載。生活援助、排泄援助、処置、リハビリ、指導、管理の6点をすべて書きます。何も一致しない場合は空白のままにしてください。
    - 生活援助: [具体的な実施内容]（例：バイタルチェック、状態観察、全身清拭、ひげそり、爪切り、水分摂取介助、精神的援助、介護者支援）
    - 排泄援助: [具体的な実施内容]（例：パウチ交換、オムツ交換、パット交換）
    - 処置: [具体的な実施内容]（例：外用薬塗布、創処置、褥瘡処置、キャビロン塗布、パッチ貼付）
    - リハビリ: [具体的な実施内容]（例：呼吸訓練、歩行訓練）
    - 指導: [具体的な実施内容]（例：観察方法指導、合併症予防指導、排便調整指導、食事指導、服薬指導、社会資源活用指導、緊急時対応指導、在宅酸素療法指導、介護指導、ＡＤＬ指導、療養指導、水分摂取指導）
    - 管理: [具体的な実施内容]（例：ストマ管理、薬物管理、在宅酸素管理）

3.【コメント】
会話に基づく今後の予定や受診予定を記載（施設名・日付など）。
"""

# Define PREVREPORTS_DIR (not defined earlier)
PREVREPORTS_DIR = Path("PrevReports")

print("✅ Configuration loaded!")
print(f"📂 Output directory: {OUT_DIR}")
print(f"📚 Knowledge base source: {PREVREPORTS_DIR}")
print(f"🤖 Models: Summary={MODEL_SUMMARY}, Embedding={MODEL_EMBED}")

print(f"🔍 RAG: Top-K={TOP_K_RETRIEVAL}, No disease constraints")
print(f"🎯 Approach: Pre-built knowledge base from all previous reports")

In [ ]:
# ============================================================================
# CELL 2: PROCESSING METER
# ============================================================================

class ProcessingMeter:
    """Track processing metrics, API calls, tokens, and costs"""
    
    def __init__(self):
        self.start_time = None
        self.end_time = None
        self.api_calls = {'summary': 0, 'embedding': 0, 'total': 0}
        self.tokens_used = {'input_tokens': 0, 'output_tokens': 0, 'embedding_tokens': 0, 'total_tokens': 0}
        self.costs = {'summary_cost': 0.0, 'embedding_cost': 0.0, 'total_cost': 0.0}
        
        # OpenAI pricing (as of 2024)
        self.pricing = {
            'gpt-4o': {'input': 2.50, 'output': 10.00},  # per 1M tokens
            'text-embedding-3-small': {'input': 0.02, 'output': 0.0},  # per 1M tokens (85% cheaper than large)
            'text-embedding-3-large': {'input': 0.13, 'output': 0.0},  # per 1M tokens
        }
    
    def start_timer(self):
        """Start timing the process"""
        self.start_time = time.time()
        print(f"🕐 Processing started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    
    def end_timer(self):
        """End timing and calculate duration"""
        self.end_time = time.time()
        duration = self.end_time - self.start_time
        print(f"⏰ Processing completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"⌛ Total duration: {self.format_duration(duration)}")
        return duration
    
    def format_duration(self, seconds):
        """Format duration in readable format"""
        hours = int(seconds // 3600)
        minutes = int((seconds % 3600) // 60)
        secs = int(seconds % 60)
        return f"{hours}h {minutes}m {secs}s"
    
    def track_api_call(self, call_type, model, input_tokens=0, output_tokens=0):
        """Track an API call and calculate cost"""
        self.api_calls[call_type] += 1
        self.api_calls['total'] += 1
        
        # Track tokens
        if call_type == 'embedding':
            self.tokens_used['embedding_tokens'] += input_tokens
        else:
            self.tokens_used['input_tokens'] += input_tokens
            self.tokens_used['output_tokens'] += output_tokens
        self.tokens_used['total_tokens'] += input_tokens + output_tokens
        
        # Calculate cost
        cost = 0.0
        if model in self.pricing:
            cost = (input_tokens / 1_000_000) * self.pricing[model]['input'] + \
                   (output_tokens / 1_000_000) * self.pricing[model]['output']
        
        # Add to appropriate cost category
        self.costs[f'{call_type}_cost'] += cost
        self.costs['total_cost'] += cost
        
        return cost
    
    def print_summary(self):
        """Print detailed summary of processing metrics"""
        duration = self.end_time - self.start_time if self.end_time else 0
        
        print(f"\n{'='*60}")
        print(f"📊 ORIGINAL RAG BASELINE PROCESSING SUMMARY")
        print(f"{'='*60}")
        
        print(f"⏱️  TIMING:")
        print(f"   • Duration: {self.format_duration(duration)}")
        
        print(f"\n📞 API CALLS:")
        for call_type, count in self.api_calls.items():
            print(f"   • {call_type.title()}: {count}")
        
        print(f"\n🔤 TOKENS USED:")
        print(f"   • Input tokens: {self.tokens_used['input_tokens']:,}")
        print(f"   • Output tokens: {self.tokens_used['output_tokens']:,}")
        print(f"   • Embedding tokens: {self.tokens_used['embedding_tokens']:,}")
        print(f"   • Total tokens: {self.tokens_used['total_tokens']:,}")
        
        print(f"\n💰 COSTS:")
        for cost_type, amount in self.costs.items():
            print(f"   • {cost_type.replace('_', ' ').title()}: ${amount:.4f}")
        
        print(f"{'='*60}")

# Global meter instance
processing_meter = ProcessingMeter()

print("✅ Processing meter ready!")


In [ ]:
# ============================================================================
# CELL 3: KNOWLEDGE BASE BUILDER & RETRIEVAL
# ============================================================================

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

class OriginalRAGKnowledgeBase:
    """
    Classic RAG: Pre-built knowledge base from all previous reports.
    Retrieves from ALL patients without disease constraints.
    """
    
    def __init__(self, api_key: str):
        self.client = AsyncOpenAI(api_key=api_key, timeout=60.0, max_retries=3)
        self.knowledge_base = []  # List of all previous visits
        self.embeddings_cache = {}
    
    async def load_from_prevreports(self, prevreports_dir: Path, patient_profiles: Dict, cache_file: Path = None):
        """Load all previous reports to build complete knowledge base"""
        
        print(f"\n{'='*60}")
        print("📚 BUILDING KNOWLEDGE BASE FROM PREVIOUS REPORTS")
        print(f"{'='*60}")
        
        # Check for cached embeddings first
        if cache_file and cache_file.exists():
            try:
                print(f"💾 Found existing embedding cache: {cache_file}")
                with open(cache_file, 'rb') as f:
                    self.knowledge_base = pickle.load(f)
                print(f"✅ Loaded {len(self.knowledge_base)} cached embeddings (no API calls needed!)")
                print(f"{'='*60}\n")
                return
            except Exception as e:
                print(f"⚠️ Error loading cache, will rebuild: {e}")
        
        if not prevreports_dir.exists():
            print(f"⚠️ PrevReports directory not found: {prevreports_dir}")
            print(f"   Absolute path: {prevreports_dir.absolute()}")
            return
        
        # Diagnostic: Show directory structure
        print(f"📂 Checking directory: {prevreports_dir.absolute()}")
        all_files = list(prevreports_dir.glob("**/*"))
        print(f"   Total files/folders found: {len(all_files)}")
        
        # Find all report CSV files (previous reports are stored in CSV format)
        report_files = list(prevreports_dir.glob("**/*.csv"))
        print(f"📄 Found {len(report_files)} CSV report files")
        
        # Show first few files for verification
        if report_files:
            print(f"   First 3 files:")
            for f in report_files[:3]:
                print(f"   - {f.relative_to(prevreports_dir)}")
        
        loaded_count = 0
        for report_file in report_files:
            try:
                # Extract patient ID from path (e.g., PrevReports/P1/PrevReports - P1.csv)
                patient_id = report_file.parent.name
                if not patient_id.startswith('P'):
                    # Try to extract from filename
                    match = re.search(r'(P\d+)', report_file.name)
                    if match:
                        patient_id = match.group(1)
                    else:
                        print(f"⚠️ Could not extract patient ID from {report_file}")
                        continue
                
                # Read CSV report file
                try:
                    report_df = pd.read_csv(report_file)
                    
                    # Extract transcription from CSV
                    # CSV has columns: Field, Value
                    # Look for "Transcription" field
                    transcription_row = report_df[report_df['Field'] == 'Transcription']
                    if transcription_row.empty:
                        print(f"⚠️ No 'Transcription' field found in {report_file}")
                        continue
                    
                    report_text = transcription_row['Value'].iloc[0]
                    
                    if pd.isna(report_text) or not str(report_text).strip():
                        print(f"⚠️ Empty transcription in {report_file}")
                        continue
                    
                    report_text = str(report_text).strip()
                    
                except Exception as csv_error:
                    print(f"⚠️ Error reading CSV {report_file}: {csv_error}")
                    continue
                
                # Get patient profile
                patient_profile = patient_profiles.get(patient_id, {
                    'patient_id': patient_id,
                    'disease': 'Unknown',
                    'gender': 'Unknown',
                    'age': 'Unknown'
                })
                
                # For previous reports, we don't have transcripts
                # So we use the report itself as the content
                await self.add_visit(
                    patient_id=patient_id,
                    transcript="",  # No transcript for historical reports
                    summary=report_text,
                    patient_profile=patient_profile
                )
                
                loaded_count += 1
                
                if loaded_count % 10 == 0:
                    print(f"   Loaded {loaded_count}/{len(report_files)} reports...")
                
            except Exception as e:
                print(f"❌ Error loading {report_file}: {e}")
                continue
        
        print(f"\n✅ Knowledge base built: {len(self.knowledge_base)} reports loaded")
        
        # Save to cache for future runs
        if cache_file:
            try:
                cache_file.parent.mkdir(parents=True, exist_ok=True)
                with open(cache_file, 'wb') as f:
                    pickle.dump(self.knowledge_base, f)
                print(f"💾 Saved embeddings to cache: {cache_file}")
                print(f"   (Next run will skip embedding generation!)")
            except Exception as e:
                print(f"⚠️ Could not save cache: {e}")
        
        print(f"{'='*60}\n")
    
    async def add_visit(self, patient_id: str, transcript: str, summary: str, patient_profile: Dict):
        """Add a visit to the knowledge base"""
        
        # Create combined text for embedding
        # For previous reports: just use the summary
        # For new visits: combine transcript + summary
        if transcript:
            combined_text = f"Patient: {patient_id}\nDisease: {patient_profile.get('disease', 'Unknown')}\nTranscript: {transcript}\nSummary: {summary}"
        else:
            combined_text = f"Patient: {patient_id}\nDisease: {patient_profile.get('disease', 'Unknown')}\nReport: {summary}"
        
        # Truncate if too long for embedding model (text-embedding-3-small has 8192 token limit)
        combined_text = self._truncate_for_embedding(combined_text, max_tokens=8000)
        
        # Generate embedding
        embedding = await self._get_embedding(combined_text)
        
        visit_data = {
            'patient_id': patient_id,
            'transcript': transcript,
            'summary': summary,
            'patient_profile': patient_profile,
            'embedding': embedding,
            'combined_text': combined_text
        }
        
        self.knowledge_base.append(visit_data)
    
    def _truncate_for_embedding(self, text: str, max_tokens: int = 8000) -> str:
        """Truncate text to fit within embedding model token limit"""
        try:
            encoding = tiktoken.encoding_for_model(MODEL_EMBED)
        except Exception:
            encoding = tiktoken.get_encoding("cl100k_base")
        
        tokens = encoding.encode(text)
        if len(tokens) <= max_tokens:
            return text
        
        # Truncate and decode
        truncated_tokens = tokens[:max_tokens]
        truncated_text = encoding.decode(truncated_tokens)
        print(f"   ⚠️ Truncated text from {len(tokens)} to {max_tokens} tokens for embedding")
        return truncated_text
    
    async def _get_embedding(self, text: str) -> np.ndarray:
        """Get embedding for text"""
        
        # Check cache
        text_hash = hash(text)
        if text_hash in self.embeddings_cache:
            return self.embeddings_cache[text_hash]
        
        # Count tokens
        try:
            encoding = tiktoken.encoding_for_model(MODEL_EMBED)
        except Exception:
            encoding = tiktoken.get_encoding("cl100k_base")
        
        input_tokens = len(encoding.encode(text))
        
        # Get embedding
        response = await self.client.embeddings.create(
            model=MODEL_EMBED,
            input=text
        )
        
        # Track API call
        processing_meter.track_api_call('embedding', MODEL_EMBED, input_tokens, 0)
        
        embedding = np.array(response.data[0].embedding)
        self.embeddings_cache[text_hash] = embedding
        
        return embedding
    
    async def retrieve_similar_visits(self, query_transcript: str, patient_profile: Dict, 
                                     top_k: int = 3) -> List[Dict]:
        """
        Retrieve similar visits from ALL patients (no disease filtering).
        Pure semantic similarity-based retrieval - classic RAG approach.
        """
        
        if not self.knowledge_base:
            print("⚠️ Knowledge base is empty")
            return []
        
        # Create query embedding
        query_text = f"Disease: {patient_profile.get('disease', 'Unknown')}\nTranscript: {query_transcript}"
        query_text = self._truncate_for_embedding(query_text, max_tokens=8000)
        query_embedding = await self._get_embedding(query_text)
        
        # Calculate similarities with ALL visits (no disease filtering)
        similarities = []
        for visit in self.knowledge_base:
            similarity = cosine_similarity(
                query_embedding.reshape(1, -1),
                visit['embedding'].reshape(1, -1)
            )[0][0]
            
            similarities.append({
                'visit': visit,
                'similarity': similarity
            })
        
        # Sort by similarity
        similarities.sort(key=lambda x: x['similarity'], reverse=True)
        
        # Return top-k
        top_visits = similarities[:top_k]
        
        print(f"🔍 Retrieved {len(top_visits)} visits from knowledge base (no disease filter)")
        for i, item in enumerate(top_visits, 1):
            visit = item['visit']
            print(f"   {i}. Patient: {visit['patient_id']}, Disease: {visit['patient_profile'].get('disease', 'Unknown')}, Similarity: {item['similarity']:.3f}")
        
        return [item['visit'] for item in top_visits]

print("✅ Original RAG knowledge base with pre-loading capability loaded!")

In [ ]:
# ============================================================================
# CELL 4: ORIGINAL RAG SUMMARY GENERATION
# ============================================================================

class OriginalRAGSummarizer:
    """Summary generation with standard RAG (no disease constraints)"""
    
    def __init__(self, api_key: str, knowledge_base: OriginalRAGKnowledgeBase):
        self.client = AsyncOpenAI(api_key=api_key, timeout=60.0, max_retries=3)
        self.knowledge_base = knowledge_base
        self.system_message = f"""あなたは経験豊富な訪問看護師です。以下の公式様式に従って、専門的で正確な看護記録を作成してください。

【公式記録様式】
{OFFICIAL_FORM_TEMPLATE}

【必須作成ルール】
1. 公式様式のすべてのセクションを必ず含めること（セクションの省略は厳禁）
2. 今回の訪問会話を最優先とし、推測や創作は禁止
3. 数値・日付・時間は会話に明記されている場合のみ記載

【記載の指針】
• 会話に出た具体的な食べ物、薬剤名、場所、時間を必ず記載
• 患者の発言や認識も重要な情報として記載
• 会話で言及がない項目は「該当なし」または「特記事項なし」
• 推測・創作・抽象的表現は一切禁止

【過去訪問記録の使用】
• 提供された過去の訪問記録は参考情報として活用可能
• ただし、今回の訪問会話が最優先
• 過去の情報を使用する場合は、時系列を明確に記載

【出力形式 - 厳守】
• 公式フォームのすべてのセクションを、構成や見出しを変更せずに出力してください。
"""
    
    async def generate_summary(self, transcript: str, patient_profile: Dict, categories: List[str]) -> Tuple[str, List[Dict]]:
        """Generate summary with retrieved context from all patients"""
        
        # Retrieve similar visits (no disease filtering)
        retrieved_visits = await self.knowledge_base.retrieve_similar_visits(
            transcript, patient_profile, top_k=TOP_K_RETRIEVAL
        )
        
        # Inject トピックス category
        enhanced_categories = list(categories) if categories else []
        if "トピックス" not in enhanced_categories:
            enhanced_categories.insert(0, "トピックス")
        
        cats_lines = "\n".join(f"【{c}】" for c in enhanced_categories)
        
        # Build context from retrieved visits
        context_parts = []
        if retrieved_visits:
            context_parts.append("【過去の訪問記録（参考情報）】")
            for i, visit in enumerate(retrieved_visits, 1):
                context_parts.append(f"\n--- 参考訪問 {i} (患者: {visit['patient_id']}, 疾患: {visit['patient_profile'].get('disease', 'Unknown')}) ---")
                context_parts.append(f"訪問記録:\n{visit['transcript'][:500]}...")  # Truncate for context window
                context_parts.append(f"記録要約:\n{visit['summary'][:500]}...")
        
        context_text = "\n".join(context_parts) if context_parts else "（過去の訪問記録なし）"
        
        # Build user prompt
        user_prompt = f"""
{context_text}

【現在の訪問記録】
{transcript}

動的カテゴリ -（以下の順序で必ず全て記載）
{cats_lines}
"""
        
        # Count tokens
        try:
            encoding = tiktoken.encoding_for_model(MODEL_SUMMARY)
        except Exception:
            encoding = tiktoken.get_encoding("o200k_base")
        
        system_tokens = len(encoding.encode(self.system_message))
        user_tokens = len(encoding.encode(user_prompt))
        input_tokens = system_tokens + user_tokens
        
        # API call
        response = await self.client.chat.completions.create(
            model=MODEL_SUMMARY,
            messages=[
                {"role": "system", "content": self.system_message},
                {"role": "user", "content": user_prompt}
            ],
            temperature=TEMP_SUMMARY,
        )
        
        # Track API call
        output_tokens = response.usage.completion_tokens if response.usage else 0
        actual_input_tokens = response.usage.prompt_tokens if response.usage else input_tokens
        processing_meter.track_api_call('summary', MODEL_SUMMARY, actual_input_tokens, output_tokens)
        
        return response.choices[0].message.content, retrieved_visits

print("✅ Original RAG summarizer loaded!")


In [ ]:
# ============================================================================
# CELL 5: FORM PROCESSING & PIPELINE
# ============================================================================

import re


def load_form_sheets(form_path: Path) -> Dict[str, pd.DataFrame]:
    """Load all sheets from an Excel form file"""
    try:
        sheets = pd.read_excel(form_path, sheet_name=None, header=None)
        print(f"✅ Loaded {len(sheets)} sheets from {form_path.name}")
        return sheets
    except Exception as e:
        print(f"❌ Error loading Excel file {form_path}: {e}")
        return {}


def extract_category_names(sheet_df: pd.DataFrame,
                          section_header: str = "利用者の状態（病状・トピックス）") -> List[str]:
    """Extract category names from form sheet"""
    if sheet_df is None or sheet_df.empty:
        return []

    # Locate section row
    idxs = sheet_df[
        sheet_df.apply(lambda row: row.astype(str).str.contains(section_header, na=False).any(), axis=1)
    ].index

    if len(idxs) == 0:
        print(f"⚠️ Section header '{section_header}' not found in sheet")
        return []

    start = idxs[0]
    cat_pat = r"【(.*?)】"
    cats: List[str] = []

    # Look for categories below the header
    for r in range(start + 1, sheet_df.shape[0]):
        row = sheet_df.iloc[r]
        if row.isna().all():
            break
        text = " ".join(row.dropna().astype(str))
        cats.extend(re.findall(cat_pat, text))

    # Normalize & dedupe
    seen = set()
    out = []
    for c in cats:
        c = c.strip()
        if c and c not in seen:
            seen.add(c)
            out.append(c)
    
    print(f"🎯 Extracted {len(out)} categories: {out}")
    return out


def map_form_to_transcript_path(form_path: Path, form_root: Path, transcript_root: Path) -> Path:
    """Map a form Excel file to its corresponding transcript file by matching filenames"""
    
    relative_path = form_path.relative_to(form_root)
    patient_folder = relative_path.parts[0] if relative_path.parts else ""
    
    # Extract patient ID
    patient_id = ""
    if patient_folder.startswith('P') and patient_folder[1:].isdigit():
        patient_id = patient_folder
    else:
        match = re.search(r'P(\d+)', form_path.name)
        if match:
            patient_id = f"P{match.group(1)}"
    
    if not patient_id:
        print(f"⚠️ Could not extract patient ID from {form_path}")
        return transcript_root / "unknown.txt"
    
    # Extract the base filename from form (without extension)
    form_stem = form_path.stem
    
    # Look for transcript with matching filename
    transcript_patient_dir = transcript_root / patient_id
    if transcript_patient_dir.exists():
        # Try to find exact match first
        matching_transcript = transcript_patient_dir / f"{form_stem}.txt"
        if matching_transcript.exists():
            return matching_transcript
        
        # Fallback: look for any txt file with similar name pattern
        txt_files = list(transcript_patient_dir.glob("*.txt"))
        for txt_file in txt_files:
            if txt_file.stem == form_stem:
                return txt_file
        
        # If no exact match found, use first file (backward compatibility)
        if txt_files:
            return txt_files[0]
    
    print(f"⚠️ No transcript found for patient {patient_id}")
    return transcript_root / patient_id / "not_found.txt"


async def process_all_forms_original_rag(
    form_root: Path,
    transcript_root: Path,
    patient_profile_csv: Path,
    prevreports_dir: Path,
    output_csv: str = "original_rag_baseline_results.csv",
    limit: int = None
) -> pd.DataFrame:
    """
    Process all forms using original RAG baseline with pre-built knowledge base.
    
    Classic RAG workflow:
    1. Load all previous reports to build complete knowledge base
    2. Process new forms with retrieval from full corpus
    3. No disease constraints - pure semantic similarity
    """
    
    processing_meter.start_timer()
    
    # Load patient profiles
    try:
        profiles_df = pd.read_csv(patient_profile_csv)
        patient_profiles = {}
        for _, row in profiles_df.iterrows():
            patient_profiles[row['patient_id']] = {
                'patient_id': row['patient_id'],
                'disease': row['disease'],
                'gender': row['gender'],
                'age': row['age']
            }
        print(f"✅ Loaded {len(patient_profiles)} patient profiles")
    except Exception as e:
        print(f"❌ Error loading patient profiles: {e}")
        return pd.DataFrame()
    
    # Initialize knowledge base
    knowledge_base = OriginalRAGKnowledgeBase(OPENAI_API_KEY)
    
    # ✨ PRE-BUILD KNOWLEDGE BASE from all previous reports (with persistent cache)
    await knowledge_base.load_from_prevreports(prevreports_dir, patient_profiles, cache_file=EMBEDDINGS_CACHE_FILE)
    
    # Initialize summarizer
    summarizer = OriginalRAGSummarizer(OPENAI_API_KEY, knowledge_base)
    
    # Find all Excel form files
    form_files = sorted(form_root.glob("*/*.xlsx"))
    print(f"📄 Found {len(form_files)} form files to process")
    
    if limit:
        form_files = form_files[:limit]
        print(f"📊 Processing limited to first {limit} files")
    
    if not form_files:
        print("⚠️ No form files found! Check FORM_ROOT path.")
        return pd.DataFrame()
    
    results = []
    
    for idx, form_path in enumerate(form_files, 1):
        print(f"\n{'='*60}")
        print(f"Processing {idx}/{len(form_files)}: {form_path.name}")
        print(f"{'='*60}")
        
        try:
            # Extract patient ID from path
            relative_path = form_path.relative_to(form_root)
            patient_folder = relative_path.parts[0] if relative_path.parts else ""
            
            patient_id = ""
            if patient_folder.startswith('P') and patient_folder[1:].isdigit():
                patient_id = patient_folder
            else:
                match = re.search(r'P(\d+)', form_path.name)
                if match:
                    patient_id = f"P{match.group(1)}"
            
            if not patient_id:
                print(f"⚠️ Could not extract patient ID from {form_path}")
                continue
            
            # Get patient profile
            patient_profile = patient_profiles.get(patient_id, {
                'patient_id': patient_id,
                'disease': 'Unknown',
                'gender': 'Unknown',
                'age': 'Unknown'
            })
            
            # Map to corresponding transcript
            transcript_path = map_form_to_transcript_path(form_path, form_root, transcript_root)
            
            if not transcript_path.exists():
                print(f"⚠️ Transcript not found: {transcript_path}")
                continue
            
            # Read transcript
            with open(transcript_path, 'r', encoding='utf-8') as f:
                transcript = f.read().strip()
            
            if not transcript:
                print(f"⚠️ Empty transcript for {patient_id}")
                continue
            
            print(f"✅ Loaded transcript: {len(transcript)} characters")
            print(f"👤 Patient: {patient_id} ({patient_profile['disease']}, {patient_profile['gender']}, {patient_profile['age']}y)")
            
            # Extract categories from form
            sheets = load_form_sheets(form_path)
            sheet1 = sheets.get("Sheet1")
            if sheet1 is None and len(sheets) > 0:
                sheet1 = list(sheets.values())[0]
            
            categories = extract_category_names(sheet1, "利用者の状態（病状・トピックス）")
            
            if not categories:
                categories = ["睡眠", "疼痛", "活動", "服薬", "水分", "栄養"]
                print(f"⚠️ No categories found, using fallback: {categories}")
            
            # Generate summary with RAG retrieval from complete knowledge base
            summary, retrieved_visits = await summarizer.generate_summary(
                transcript=transcript,
                patient_profile=patient_profile,
                categories=categories
            )
            
            # Store result
            results.append({
                'Patient ID': patient_id,
                'Form File': form_path.name,
                'Transcript File': transcript_path.name,
                'Categories': ', '.join(categories),
                'Disease': patient_profile['disease'],
                'Retrieved From': '; '.join([f"{v['patient_id']}({v['patient_profile'].get('disease', 'Unknown')})" for v in retrieved_visits]),
                'Summary': summary
            })
            
            print(f"✅ Generated summary: {len(summary)} characters")
            print(f"📊 Cumulative: {processing_meter.api_calls['total']} API calls, {processing_meter.tokens_used['total_tokens']:,} tokens, ${processing_meter.costs['total_cost']:.4f}")
            
        except Exception as e:
            print(f"❌ Error processing {form_path.name}: {str(e)}")
            import traceback
            traceback.print_exc()
            continue
    
    # End timer
    processing_meter.end_timer()
    
    # Create DataFrame and save
    df_results = pd.DataFrame(results)
    
    if not df_results.empty:
        output_path = OUT_DIR / output_csv
        df_results.to_csv(output_path, index=False, encoding='utf-8-sig')
        print(f"\n✅ Saved {len(df_results)} results to {output_path}")
    else:
        print("\n⚠️ No results to save")
    
    # Final meter report
    processing_meter.print_summary()
    
    return df_results

print("✅ Original RAG pipeline with pre-built knowledge base loaded!")
print("   • load_from_prevreports() - Build knowledge base from all previous reports")
print("   • process_all_forms_original_rag() - Generate summaries with full RAG retrieval")

In [ ]:
# ============================================================================
# CELL 6: RUN PROCESSING
# ============================================================================

# Run original RAG baseline processing with pre-built knowledge base
# Set limit=None to process all forms, or limit=10 for testing

results = await process_all_forms_original_rag(
    form_root=FORM_ROOT,
    transcript_root=TRANSCRIPT_ROOT,
    patient_profile_csv=PATIENT_PROFILE_CSV,
    prevreports_dir=PREVREPORTS_DIR,
    output_csv="original_rag_baseline_results.csv"
)

print("\n" + "="*80)
print("🎉 ORIGINAL RAG BASELINE PROCESSING COMPLETE")
print("="*80)
print(f"\n📊 Processed {len(results)} summaries")
print(f"📂 Results saved to: {OUT_DIR}")
print(f"\n✨ Approach: Classic RAG with pre-built knowledge base")
print(f"🔍 Retrieval: Semantic similarity from ALL previous reports (no disease filtering)")
print(f"📚 Knowledge base: Built from PrevReports directory")